伪量化 非对称量化 正确率测试

In [1]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F

# ==========================================
# 1. 核心：手工伪量化函数 (模拟硬件的截断误差)
# ==========================================
def fake_quantize(tensor, q_min=-128, q_max=127):
    """
    对输入的 Tensor 进行 INT8 伪量化：
    Float -> 算出 Scale/ZP -> 截断为 INT8 -> 还原为带误差的 Float
    """
    with torch.no_grad():
        r_min = tensor.min().item()
        r_max = tensor.max().item()
        
        # 避免全 0 张量导致除以 0
        if r_min == r_max:
            return tensor
            
        # 1. 计算 Scale 和 Zero Point
        scale = (r_max - r_min) / (q_max - q_min)
        zero_point = round(q_min - (r_min / scale))
        zero_point = max(q_min, min(q_max, zero_point)) # 限制在 INT8 范围内
        
        # 2. 量化 (模拟硬件的 INT8)
        q_tensor = torch.round(tensor / scale + zero_point)
        q_tensor = torch.clamp(q_tensor, q_min, q_max)
        
        # 3. 反量化 (变回 Float，但此时已经丢失了精度)
        dq_tensor = (q_tensor - zero_point) * scale
        
        return dq_tensor

# ==========================================
# 2. 带有量化模拟特性的 CNN (继承你之前的架构)
# ==========================================
class PTQ_HardwareFriendlyCNN(nn.Module):
    def __init__(self):
        super(PTQ_HardwareFriendlyCNN, self).__init__()
        # 定义原始结构
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.fc = nn.Linear(32 * 8 * 8, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        # 模拟硬件：首先对输入的图片进行 INT8 伪量化
        x = fake_quantize(x)

        # ====== 第一层 Conv1 ======
        # 对权重和偏置进行伪量化
        w1_fq = fake_quantize(self.conv1.weight)
        b1_fq = fake_quantize(self.conv1.bias) if self.conv1.bias is not None else None
        
        # 执行卷积
        x = F.conv2d(x, w1_fq, b1_fq, stride=self.conv1.stride, padding=self.conv1.padding)
        # 模拟激活和池化层输入前的量化截断 (因为硬件这里会把 INT32 重新量化为 INT8)
        x = fake_quantize(x) 
        x = self.pool(self.relu(x))

        # ====== 第二层 Conv2 ======
        w2_fq = fake_quantize(self.conv2.weight)
        b2_fq = fake_quantize(self.conv2.bias) if self.conv2.bias is not None else None
        
        x = F.conv2d(x, w2_fq, b2_fq, stride=self.conv2.stride, padding=self.conv2.padding)
        x = fake_quantize(x)
        x = self.pool(self.relu(x))

        # ====== 全连接层 FC ======
        x = torch.flatten(x, 1)
        x = fake_quantize(x) # Flatten 后的特征传入 FC 前再次截断
        
        w_fc_fq = fake_quantize(self.fc.weight)
        b_fc_fq = fake_quantize(self.fc.bias) if self.fc.bias is not None else None
        
        x = F.linear(x, w_fc_fq, b_fc_fq)
        # 输出层通常保留高精度(如INT32)供 CPU 找最大值，所以这里不强制伪量化
        return x

# ==========================================
# 3. 运行验证评估
# ==========================================
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # 1. 准备测试数据 (注意：不再需要打乱)
    transform = transforms.Compose([transforms.ToTensor()])
    testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=False, transform=transform)
    testloader = torch.utils.data.DataLoader(testset, batch_size=1000, shuffle=False)

    # 2. 加载原始 FP32 权重到伪量化模型中
    model = PTQ_HardwareFriendlyCNN().to(device)
    # ！！请将这里替换为你实际保存的文件名 ！！
    model_path = 'my_hardware_cnn_fp32_69p.pth' 
    
    try:
        # strict=False 允许我们灵活加载权重
        model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True), strict=False)
        print(f"成功加载模型权重: {model_path}")
    except Exception as e:
        print(f"加载模型失败，请检查文件名和路径。错误信息: {e}")
        return

    model.eval()

    # 3. 在测试集上跑一边伪量化推理
    print("开始进行 INT8 PTQ 伪量化准确率测试 (这可能需要几秒钟)...")
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            # 这里前向传播时，会自动触发我们在 forward 里写的 fake_quantize 逻辑
            outputs = model(inputs) 
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"==================================================")
    print(f"🔥 FP32 原始准确率目标：~69.33 %")
    print(f"📉 INT8 PTQ 量化后准确率：{accuracy:.2f} %")
    print(f"==================================================")

if __name__ == '__main__':
    main()

Using device: cuda
成功加载模型权重: my_hardware_cnn_fp32_69p.pth
开始进行 INT8 PTQ 伪量化准确率测试 (这可能需要几秒钟)...
🔥 FP32 原始准确率目标：~69.33 %
📉 INT8 PTQ 量化后准确率：69.44 %
